In [2]:
from autogen_agentchat.agents import AssistantAgent,SocietyOfMindAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()

True

In [3]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
model_client=OpenAIChatCompletionClient(model='gpt-4o')

In [6]:
async def main(task : str)-> None:
    agent_1=AssistantAgent(
        name="AssistantAgent_1",
        model_client=model_client,
        description="An Agent who writes short story on any subject.",
        system_message="""You are writer, you writes short story very well. 
        Write a short story within 50 words on the topic which user asked."""
    )
    agent_2=AssistantAgent(
        name="AssistantAgent_2",
        model_client=model_client,
        description=" An editor agent",
        system_message="""You are an editor, provide critical feedback within 30 words. 
        Respond with 'APPROVE' if the text addresses all feedbacks."""
    )

    inner_termination=TextMentionTermination("APPROVE")
    inner_team=RoundRobinGroupChat(participants=[agent_1,agent_2],termination_condition=inner_termination)

    society_of_mind_agent=SocietyOfMindAgent(name="Society_of_mind",
                                             team=inner_team,model_client=model_client,
                                             response_prompt="""Output a standalone response to the 
                                             original request, without mentioning any of the 
                                             intermediate discussion response the final 
                                             response within 50 words""")
    
    agent_3=AssistantAgent(
        name="AssistantAgent_3",
        model_client=model_client,
        description="Language translator agent",
        system_message="""You are a translator who translate input text into 
        bengali language within 50 words. After completing your task raise 'TERMINATE' to exit.""")
    
    termination=TextMentionTermination('TERMINATE')
    team=RoundRobinGroupChat(participants=[society_of_mind_agent,agent_3],
                             max_turns=2,
                             termination_condition=termination)
    
    stream=team.run_stream(task=task)

    await Console(stream=stream)

    

In [7]:
await main(task="Write a story with suspances.")

---------- TextMessage (user) ----------
Write a story with suspances.


---------- TextMessage (AssistantAgent_1) ----------
In the dim-lit attic, Emily uncovered a dusty, forgotten box. Inside, she found a key with an ominous note: "Unlock only if you dare." Curiosity piqued, she followed the note's directions, leading to a hidden cellar door. As she turned the key, a chilling whisper echoed, "Welcome back, Emily."
---------- TextMessage (AssistantAgent_2) ----------
Consider adding more background on Emily's character and motivations to enhance the suspense. Elaborate on the setting to build atmosphere and provide a captivating resolution to the mystery.
---------- TextMessage (AssistantAgent_1) ----------
In the dim-lit attic, Emily uncovered a dusty, forgotten box. Inside, she found a key with an ominous note: "Unlock only if you dare." Curiosity piqued, she followed the note's directions, leading to a hidden cellar door. As she turned the key, a chilling whisper echoed, "Welcome back, Emily."
---------- TextMessage (AssistantAgent_2) ----------
Consid